# Tier 0: Data Audit & Exploration

**Goal:** Understand what's in the dataset before diving into analysis.

This notebook will help you:
1. Connect to the SQLite database
2. Run basic queries to understand the data
3. Identify data quality issues
4. Document findings

---

## 1. Setup & Connection

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Connect to database
conn = sqlite3.connect('../db/airbnb.db')

# Set up plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Connected to database!")

## 2. Basic Counts

In [ ]:
# How many records in each table?
for table in ['listings', 'calendar', 'reviews']:
    count = pd.read_sql(f'SELECT COUNT(*) as cnt FROM {table}', conn)['cnt'].iloc[0]
    print(f"{table:12} {count:>12,} rows")

## 3. Listings by Borough

In [ ]:
# Listings per borough
borough_counts = pd.read_sql("""
    SELECT neighbourhood_group_cleansed as borough, COUNT(*) as count
    FROM listings
    GROUP BY neighbourhood_group_cleansed
    ORDER BY count DESC
""", conn)

print(borough_counts)

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(borough_counts['borough'], borough_counts['count'])
ax.set_title('Number of Listings by Borough')
ax.set_ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../output/charts/listings_by_borough.png', dpi=150)
plt.show()

## 4. Room Type Distribution

In [ ]:
# Room type distribution
room_types = pd.read_sql("""
    SELECT room_type, COUNT(*) as count,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM listings), 1) as pct
    FROM listings
    GROUP BY room_type
    ORDER BY count DESC
""", conn)

print(room_types)

# Pie chart
fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(room_types['count'], labels=room_types['room_type'], autopct='%1.1f%%')
ax.set_title('Room Type Distribution')
plt.savefig('../output/charts/room_type_distribution.png', dpi=150)
plt.show()

## 5. Price Analysis

In [ ]:
# Price statistics by borough
price_stats = pd.read_sql("""
    SELECT neighbourhood_group_cleansed as borough,
           COUNT(*) as listings,
           ROUND(MIN(price_numeric), 2) as min_price,
           ROUND(AVG(price_numeric), 2) as avg_price,
           ROUND(MAX(price_numeric), 2) as max_price
    FROM listings
    WHERE price_numeric > 0 AND price_numeric < 1000
    GROUP BY neighbourhood_group_cleansed
    ORDER BY avg_price DESC
""", conn)

print(price_stats)

# Price distribution histogram
prices = pd.read_sql("""
    SELECT price_numeric FROM listings
    WHERE price_numeric > 0 AND price_numeric < 500
""", conn)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(prices['price_numeric'], bins=50, edgecolor='black')
ax.set_title('Price Distribution (Under $500/night)')
ax.set_xlabel('Price ($)')
ax.set_ylabel('Count')
plt.savefig('../output/charts/price_distribution.png', dpi=150)
plt.show()

## 6. Review Analysis

In [ ]:
# Listings with zero reviews
zero_reviews = pd.read_sql("""
    SELECT COUNT(*) as zero_review_count,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM listings), 1) as pct
    FROM listings
    WHERE number_of_reviews = 0
""", conn)

print(f"Listings with 0 reviews: {zero_reviews['zero_review_count'].iloc[0]:,} ({zero_reviews['pct'].iloc[0]}%)")

# Review date range
review_dates = pd.read_sql("""
    SELECT MIN(date) as earliest, MAX(date) as latest,
           COUNT(DISTINCT listing_id) as listings_with_reviews
    FROM reviews
""", conn)

print(f"\nReview date range: {review_dates['earliest'].iloc[0]} to {review_dates['latest'].iloc[0]}")
print(f"Listings with at least 1 review: {review_dates['listings_with_reviews'].iloc[0]:,}")

## 7. Revenue Estimates (Pre-calculated)

In [ ]:
# Revenue by borough (using pre-calculated field)
revenue_by_borough = pd.read_sql("""
    SELECT neighbourhood_group_cleansed as borough,
           COUNT(*) as listings,
           ROUND(AVG(estimated_revenue_l365d), 0) as avg_revenue,
           ROUND(AVG(estimated_occupancy_l365d), 0) as avg_occupancy
    FROM listings
    WHERE estimated_revenue_l365d > 0
    GROUP BY neighbourhood_group_cleansed
    ORDER BY avg_revenue DESC
""", conn)

print(revenue_by_borough)

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(revenue_by_borough['borough'], revenue_by_borough['avg_revenue'])
axes[0].set_title('Average Estimated Revenue by Borough')
axes[0].set_ylabel('Revenue ($)')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(revenue_by_borough['borough'], revenue_by_borough['avg_occupancy'])
axes[1].set_title('Average Occupancy by Borough')
axes[1].set_ylabel('Days Booked')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../output/charts/revenue_by_borough.png', dpi=150)
plt.show()

## 8. Data Quality Check

In [ ]:
# Check for missing values in key columns
key_columns = ['price', 'room_type', 'neighbourhood_cleansed', 
               'latitude', 'longitude', 'number_of_reviews',
               'estimated_revenue_l365d']

for col in key_columns:
    null_count = pd.read_sql(f"""
        SELECT COUNT(*) as cnt FROM listings
        WHERE {col} IS NULL OR {col} = ''
    """, conn)['cnt'].iloc[0]
    print(f"{col:30} {null_count:>8,} nulls")

## 9. Your Findings

**TODO:** Write a summary of what you learned from this audit.

Questions to answer:
1. How many listings are there per borough?
2. What's the date range of the reviews?
3. What percentage of listings have zero reviews?
4. What are the room type distributions?
5. Are there any data quality issues?

Write your answers below:

In [ ]:
# Your analysis here
print("TODO: Write your findings summary")

---

## Next Steps

Once you've completed this audit:
1. Document any data quality issues you found
2. Discuss findings with your team
3. Move on to Tier 1: Market Overview

**Remember:** Don't skip this step! Understanding the data is crucial before doing any analysis.